# Feature Engineering - Match Prediction Features

**Project:** Football Match Outcome Prediction  
**Previous:** 01_exploratory_data_analysis.ipynb  
**Date:** January 2026

## Objective
Create predictive features based on insights from EDA. Focus on features that capture team strength, momentum, and match context.

## Feature Strategy
Based on EDA findings:
1. **Team Strength:** ELO ratings
2. **Momentum:** Recent form, win streaks
3. **Performance:** Goal scoring/conceding patterns
4. **Context:** Home/away, season timing, rest days
5. **History:** Head-to-head records

In [2]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import sqlite3
import sys
# Import only engineer_features (we'll use SQL for loading)
sys.path.append('..')
from utils import engineer_features
sns.set_style('whitegrid')
%matplotlib inline
print("✅ Libraries loaded")

✅ Libraries loaded


## 1. Load Base Data

In [ ]:
# Load data using SQL
db_path = '../data/database/football.db'
conn = sqlite3.connect(db_path)
query = """
    SELECT *
    FROM matches
    WHERE status = 'FINISHED'
    ORDER BY utc_date
"""
df = pd.read_sql_query(query, conn)
conn.close()
df['utc_date'] = pd.to_datetime(df['utc_date'])
df['total_goals'] = df['home_score'] + df['away_score']
print(f"✅ Loaded {len(df):,} matches using SQL")
print(f"Date range: {df['utc_date'].min().date()} to {df['utc_date'].max().date()}")

## 2. Feature Engineering Pipeline

We'll create 15 features using our utility function, then validate each one.

In [ ]:
# Engineer all features
df_features = engineer_features(df)

print("✅ Features engineered")
print(f"\nDataset shape: {df_features.shape}")
print(f"\nNew feature columns added:")
new_cols = [col for col in df_features.columns if col not in df.columns]
for i, col in enumerate(new_cols, 1):
    print(f"  {i}. {col}")

## 3. Feature Validation

### 3.1 ELO Ratings

In [ ]:
# ELO distribution
print("ELO Rating Statistics:")
print(df_features[['home_elo', 'away_elo', 'elo_diff']].describe())

# Visualize
fig,ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].hist(df_features['elo_diff'], bins=50, color='#667eea', edgecolor='black', alpha=0.7)
ax[0].axvline(0, color='red', linestyle='--', label='Equal strength')
ax[0].set_title('ELO Difference Distribution')
ax[0].set_xlabel('Home ELO - Away ELO')
ax[0].legend()

# ELO vs outcome
result_map = {'HOME_TEAM': 1, 'H': 1, 'DRAW': 0, 'D': 0, 'AWAY_TEAM': -1, 'A': -1}
df_features['outcome_numeric'] = df_features['winner'].map(result_map)

ax[1].scatter(df_features['elo_diff'], df_features['outcome_numeric'], alpha=0.1)
ax[1].set_title('ELO Difference vs Match Outcome')
ax[1].set_xlabel('ELO Difference')
ax[1].set_ylabel('Outcome (1=Home, 0=Draw, -1=Away)')

plt.tight_layout()
plt.show()

print(f"\nCorrelation with outcome: {df_features['elo_diff'].corr(df_features['outcome_numeric']):.3f}")

**Validation:** Positive correlation - teams with higher ELO tend to win ✓

### 3.2 Recent Form

In [ ]:
# Form statistics
print("Recent Form Statistics:")
print(df_features[['home_form', 'away_form', 'form_diff']].describe())

# Form vs outcome
fig = px.box(df_features, x='winner', y='form_diff',
             title='Form Difference by Match Outcome',
             labels={'winner': 'Match Outcome', 'form_diff': 'Form Difference (pts/match)'})
fig.show()

print(f"\nCorrelation with outcome: {df_features['form_diff'].corr(df_features['outcome_numeric']):.3f}")

**Validation:** Strong correlation - form is our best predictor! ✓

### 3.3 Goal Statistics

In [ ]:
# Goal averages
print("Goal Average Statistics:")
print(df_features[['home_goals_avg', 'away_goals_avg', 
                     'home_conceded_avg', 'away_conceded_avg']].describe())

# Attacking vs defensive strength
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(df_features['home_goals_avg'], df_features['home_score'], alpha=0.3)
ax1.set_title('Home Goals Avg vs Actual Goals Scored')
ax1.set_xlabel('Rolling Average (last 5)')
ax1.set_ylabel('Actual Goals in Match')

ax2.scatter(df_features['away_conceded_avg'], df_features['home_score'], alpha=0.3)
ax2.set_title('Away Defense vs Home Goals Scored')
ax2.set_xlabel('Away Conceded Avg')
ax2.set_ylabel('Home Goals')

plt.tight_layout()
plt.show()

**Validation:** Recent goal patterns predict current match scoring ✓

### 3.4 Win Streaks & Clean Sheets

In [ ]:
# Streak statistics
print("Win Streak Statistics:")
print(df_features[['home_win_streak', 'away_win_streak']].describe())

print("\nClean Sheet Statistics:")
print(df_features[['home_clean_sheets', 'away_clean_sheets']].describe())

# Win rate by streak length
streak_bins = [0, 1, 2, 3, 10]
df_features['streak_category'] = pd.cut(df_features['home_win_streak'], bins=streak_bins)
win_rate = df_features[df_features['winner'].isin(['HOME_TEAM', 'H'])].groupby('streak_category').size() / df_features.groupby('streak_category').size()

print("\nHome win rate by streak:")
print(win_rate)

**Validation:** Teams on winning streaks have higher win probability ✓

### 3.5 Contextual Features

In [ ]:
# Rest days
print("Rest Days Statistics:")
print(df_features['rest_days'].describe())

# December effect
dec_games = df_features[df_features['is_december'] == 1]
other_games = df_features[df_features['is_december'] == 0]

print(f"\nDecember avg goals: {dec_games['total_goals'].mean():.2f}")
print(f"Other months avg goals: {other_games['total_goals'].mean():.2f}")

# Season progress
fig = px.scatter(df_features, x='season_progress', y='total_goals',
                 title='Goals vs Season Progress',
                 labels={'season_progress': 'Season Progress (0=start, 1=end)'})
fig.show()

## 4. Feature Correlation Analysis

In [ ]:
# Select features for modeling
features = [
    'elo_diff', 'form_diff', 'home_win_streak', 'away_win_streak',
    'home_clean_sheets', 'away_clean_sheets', 'home_goals_avg', 'away_goals_avg',
    'home_conceded_avg', 'away_conceded_avg', 'h2h_home_wins', 'h2h_away_wins',
    'rest_days', 'season_progress', 'is_december'
]

# Correlation matrix
corr_matrix = df_features[features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

# Check for high multicollinearity
high_corr = np.where(np.abs(corr_matrix) > 0.8)
high_corr_pairs = [(corr_matrix.index[x], corr_matrix.columns[y], corr_matrix.iloc[x, y]) 
                   for x, y in zip(*high_corr) if x != y and x < y]

if high_corr_pairs:
    print("\n⚠️ Highly correlated features:")
    for feat1, feat2, corr in high_corr_pairs:
        print(f"  {feat1} <-> {feat2}: {corr:.3f}")
else:
    print("\n✓ No highly correlated features (< 0.8)")

## 5. Feature vs Target Correlation

In [ ]:
# Correlation with match outcome
correlations = df_features[features].corrwith(df_features['outcome_numeric']).sort_values(ascending=False)

fig = px.bar(x=correlations.index, y=correlations.values,
             title='Feature Correlation with Match Outcome',
             labels={'x': 'Feature', 'y': 'Correlation'})
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.show()

print("\nTop 5 Predictive Features:")
for feat, corr in correlations.head(5).items():
    print(f"  {feat}: {corr:.3f}")

## 6. Save Engineered Features

In [ ]:
# Save feature-engineered dataset
output_cols = ['utc_date', 'season', 'home_team_name', 'away_team_name', 
               'home_score', 'away_score', 'winner'] + features

df_features[output_cols].to_csv('engineered_features.csv', index=False)
print("✅ Features saved to engineered_features.csv")
print(f"\nShape: {df_features[output_cols].shape}")
print(f"Features: {len(features)}")

## 7. Summary

### Features Created: 15

**Strength (2):**
- `elo_diff` - Team strength differential
- `form_diff` - Recent performance (MOST IMPORTANT)

**Momentum (4):**
- `home_win_streak`, `away_win_streak` - Hot/cold teams
- `home_clean_sheets`, `away_clean_sheets` - Defensive momentum

**Performance (4):**
- `home_goals_avg`, `away_goals_avg` - Attacking strength
- `home_conceded_avg`, `away_conceded_avg` - Defensive strength

**History (2):**
- `h2h_home_wins`, `h2h_away_wins` - Head-to-head record

**Context (3):**
- `rest_days` - Fatigue factor
- `season_progress` - Early/late season
- `is_december` - Fixture congestion

### Validation Results ✓
- All features show logical relationship with outcomes
- No multicollinearity issues (all < 0.8)
- Form is strongest single predictor
- Ready for modeling!

---

**Next Step:** Model development (03_model_development.ipynb)